Week 6 Spark Assignment
Apache Spark Data Processing
Objective: Understand Spark architecture and perform efficient data processing using transformations, filtering, schema handling, and optimized file formats. Steps: Understand Spark architecture (Driver, Cluster Manager, Executors) and execution modes. Learn Lazy Evaluation and how it optimizes execution using DAG (Lineage Graph). Read data from files (CSV, Parquet) with proper schema handling. Perform filtering and selection of required columns. Modify DataFrames (rename columns, cast data types, add new columns). Apply transformations and actions appropriately. Understand wide transformations and performance concepts (Shuffle, Predicate Pushdown). Work with different file formats (CSV vs Parquet) and their impact on performance. Handle null values and filter datasets efficiently. Build data pipelines (read → transform → filter → write). Save processed data into required formats (CSV/Parquet). Follow best practices for large datasets (avoid collect(), use show()). Output: Spark code (PySpark/Scala) + execution results + brief insights on performance and architecture.

In [0]:
from pyspark.sql import SparkSession
spark=SparkSession.builder \
    .appName("Week6_Spark_Assignment") \
    .getOrCreate()


Q1: Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application. 

Ans:
Driver:The Driver is the main process of a spark application.
It is responsible for craeting SparkSession and SparkContext,converting user code into execution plan,creation of DAG,scheduling jobs and tasks,communicating with cluster manager,collecting results from executors.
Cluster Manager:Manages resources required by spark applications.It is responsible for allocating cpu and memory resources,starts executors on worker nodes,manages cluster resources.
Executor:it is a jvm process running on worker nodes.It executes tasks assigned by driver,performs data processing,stores cached data,sends results back to driver.
So the spark architecture follows:application->Driver->SparkContext->Cluster Manager->Worker nodes->Executors->Tasks.


Q2: How does Spark’s Lazy Evaluation strategy improve performance when chain-processing large datasets? 


Ans:
Lazy Evaluation is an optimisation technique in Apache Spark where transformations are not executed immediately when they are called.Instead of executing each transformation step by step,Spark builds a logical execution plan called a DAG.The actual execution starts only when an action is triggered.
So lazy evalauation improves performance :
1.Query optimisation:
Spark analyses the complete chain of transformations before execution.This allows sparks catalyst optimiser to create an optimised execution plan.
2.Avoids unneccessary computations:Spark doesnt execute imtermediate transformations immediately.Only the final required result is calculated.
3.Reduces disk  and memory usage:Since spark combines multiple transformations into a single optimised plan,  unnecessary intermediate data storage is avoided.
4.supports DAG Optimisation:All transformations are stored in a DAG.
when an action occurs:DAG->Stages->Tasks->Executors.

Q3: Write a Spark command to read a CSV file located at "data/source.csv", ensuring the first row is treated as a header and inferSchema is enabled.

In [0]:
df=spark.table("workspace.default.source")
df.show(10)

+--------+-------+----------+------------+--------------+---------+----------+--------+---------+---------+------+--------+-------------+----------+------------+
|order_id|user_id|product_id|    old_name|      category|    price|base_price|quantity|   amount|   status|region|priority|customer_name|order_date|discount_pct|
+--------+-------+----------+------------+--------------+---------+----------+--------+---------+---------+------+--------+-------------+----------+------------+
|  100001|  20001|     P1001|       Novel|         Books|  1299.29|   1299.29|       4|  5197.16|  Pending| North|     Low|  Pooja Verma|2025-03-08|           0|
|  100002|  20002|     P1002|      Laptop|   Electronics| 10970.97|  10970.97|       1| 10970.97|  Pending|  West|    High| Rohan Sharma|2024-06-12|          15|
|  100003|  20003|     P1003|Coffee Beans|       Grocery| 13929.67|  13929.67|       6| 83578.02|Completed| North|  Medium|  Priya Singh|2024-09-27|          20|
|  100004|  20004|     P1004

In [0]:
df.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- user_id: long (nullable = true)
 |-- product_id: string (nullable = true)
 |-- old_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: string (nullable = true)
 |-- base_price: double (nullable = true)
 |-- quantity: long (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- discount_pct: long (nullable = true)



In [0]:
df_csv=spark.read \
.option("header",True) \
.option("inferSchema",True) \
.csv("data/source.csv")

In [0]:
df = spark.table(
    "workspace.default.source"
)

df.show(10)

+--------+-------+----------+------------+--------------+---------+----------+--------+---------+---------+------+--------+-------------+----------+------------+
|order_id|user_id|product_id|    old_name|      category|    price|base_price|quantity|   amount|   status|region|priority|customer_name|order_date|discount_pct|
+--------+-------+----------+------------+--------------+---------+----------+--------+---------+---------+------+--------+-------------+----------+------------+
|  100001|  20001|     P1001|       Novel|         Books|  1299.29|   1299.29|       4|  5197.16|  Pending| North|     Low|  Pooja Verma|2025-03-08|           0|
|  100002|  20002|     P1002|      Laptop|   Electronics| 10970.97|  10970.97|       1| 10970.97|  Pending|  West|    High| Rohan Sharma|2024-06-12|          15|
|  100003|  20003|     P1003|Coffee Beans|       Grocery| 13929.67|  13929.67|       6| 83578.02|Completed| North|  Medium|  Priya Singh|2024-09-27|          20|
|  100004|  20004|     P1004

In [0]:
df.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- user_id: long (nullable = true)
 |-- product_id: string (nullable = true)
 |-- old_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: string (nullable = true)
 |-- base_price: double (nullable = true)
 |-- quantity: long (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- discount_pct: long (nullable = true)



Q4: What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar) and why does it matter for performance? 

Ans:
CSV (Comma Separated Values) is a row-oriented text file format.
In row-based storage, all values belonging to a single record are stored together.
 Example Table:

|order_id|product|category|price|
|101|Laptop|Electronics|70000|
|102|Mobile|Electronics|30000|
|103|Chair|Furniture|5000|

How CSV stores data:
Row 1:
101, Laptop, Electronics, 70000
Row 2:
102, Mobile, Electronics, 30000
Row 3:
103, Chair, Furniture, 5000
Each complete row is stored together.
characteristics:
- Stores data row by row
- Text-based format
- Human readable
- Does not store schema information
- Provides less compression
- Requires more storage space
- Slower for analytical processing

 Parquet (Columnar Storage)
Parquet is a column-oriented binary file format designed for big data processing.
In columnar storage, values belonging to the same column are stored together.
How Parquet stores data:
order_id column:
101
102
103
product column:
Laptop
Mobile
Chair
category column:
Electronics
Electronics
Furniture
price column:
70000
30000
5000
Each column is stored separately.
Characteristics of Parquet:
- Stores data column-wise
- Binary file format
- Stores schema metadata
- Provides high compression
- Supports predicate pushdown
- Optimized for analytical queries
- Faster with large datasets.

The storage format decides how much data Spark needs to read and process.

Spark performance depends on:

- Amount of data scanned
- Disk Input/Output (I/O)
- Memory usage
- Network transfer
- Query execution time
Therefore, Parquet is preferred in Apache Spark because columnar storage reduces data scanning, improves compression, saves memory, and provides faster performance for large-scale analytical workloads.



Q5: Given a DataFrame df, write a query to select the columns product_id and price where the category is 'Electronics'. 

In [0]:
electronics_df=df.filter(
    df.category=="Electronics"
).select(
    "product_id",
    "price"
)
electronics_df.show()

+----------+--------+
|product_id|   price|
+----------+--------+
|     P1002|10970.97|
|     P1004|22998.27|
|     P1005| 33080.1|
|     P1010|19111.88|
|     P1015|11273.63|
|     P1019|40410.71|
|     P1022|23865.59|
|     P1045|41652.91|
|     P1059|26816.41|
|     P1065|22552.66|
|     P1071|12657.32|
|     P1075|38645.12|
|     P1078|    NULL|
|     P1080|31576.49|
|     P1083| 23289.4|
|     P1091|41201.57|
|     P1092| 44368.6|
|     P1112| 8635.07|
|     P1113|43532.69|
|     P1129|35749.48|
+----------+--------+
only showing top 20 rows


Q6: Write the code to "revise" a DataFrame by renaming the column old_name to new_name and casting the price column from a String to a Double. 

In [0]:
from pyspark.sql.functions import col, regexp_replace

revised_df=df.withColumnRenamed(
    "old_name",
    "new_name"
).withColumn(
    "price",
    regexp_replace(
        col("price"),
        "\\$",
        ""
    ).cast("double")
)

revised_df.show(10)

+--------+-------+----------+------------+--------------+--------+----------+--------+---------+---------+------+--------+-------------+----------+------------+
|order_id|user_id|product_id|    new_name|      category|   price|base_price|quantity|   amount|   status|region|priority|customer_name|order_date|discount_pct|
+--------+-------+----------+------------+--------------+--------+----------+--------+---------+---------+------+--------+-------------+----------+------------+
|  100001|  20001|     P1001|       Novel|         Books| 1299.29|   1299.29|       4|  5197.16|  Pending| North|     Low|  Pooja Verma|2025-03-08|           0|
|  100002|  20002|     P1002|      Laptop|   Electronics|10970.97|  10970.97|       1| 10970.97|  Pending|  West|    High| Rohan Sharma|2024-06-12|          15|
|  100003|  20003|     P1003|Coffee Beans|       Grocery|13929.67|  13929.67|       6| 83578.02|Completed| North|  Medium|  Priya Singh|2024-09-27|          20|
|  100004|  20004|     P1004|     

Q7: How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails? 

Ans:
DAG (Directed Acyclic Graph) represents the sequence of transformations performed on data.
Spark does not execute transformations immediately. It stores all transformation steps in a lineage graph.
Example:
Read Data->filter()->select()->result
Fault Tolerance using DAG
In Spark, data is divided into partitions and processed by Worker Nodes.
If a Worker Node fails:
- Spark checks the DAG lineage graph.
- It identifies how the lost partition was created.
- Spark recomputes only the missing partition using the original data and transformations.
- The recovered task is assigned to another available Worker Node.
Example:
df1=Read CSV->df2 = filter()->df3 = select()
If df3 data is lost, Spark uses lineage:
Read CSV → filter() → select() and recreates the lost data.

- Automatic failure recovery
- No need to store duplicate intermediate data
- Saves memory and storage
- Improves reliability for large datasets

Therefore, DAG lineage provides fault tolerance by tracking transformations and rebuilding lost partitions when failures occur.

Q8: Write a query to filter a DataFrame df_orders for rows where the status is 'Completed' AND the amount is greater than 1000. 

In [0]:
df_orders=df.filter(
    (df.status=="Completed") &
    (df.amount>1000)
)

df_orders.show(10)

+--------+-------+----------+------------+--------------+--------+----------+--------+---------+---------+------+--------+-------------+----------+------------+
|order_id|user_id|product_id|    old_name|      category|   price|base_price|quantity|   amount|   status|region|priority|customer_name|order_date|discount_pct|
+--------+-------+----------+------------+--------------+--------+----------+--------+---------+---------+------+--------+-------------+----------+------------+
|  100003|  20003|     P1003|Coffee Beans|       Grocery|13929.67|  13929.67|       6| 83578.02|Completed| North|  Medium|  Priya Singh|2024-09-27|          20|
|  100004|  20004|     P1004|      Camera|   Electronics|22998.27|  22998.27|       7|160987.89|Completed|  East|     Low|  Priya Singh|2024-07-15|          15|
|  100012|   NULL|     P1012|     Toaster|Home & Kitchen|26991.98|  26991.98|       9|242927.82|Completed|  East|  Medium|   Priya Iyer|2024-09-02|           0|
|  100013|  20013|     P1013|     

Q9: Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory. 

Ans:
Predicate Pushdown is a Spark optimization technique where filter conditions are pushed closer to the data source before loading the data into memory.
Instead of reading the complete dataset first and then applying filters, Spark tries to skip unnecessary data during the reading stage itself.
working in Parquet
Parquet stores data in columnar format along with metadata such as:
- Column information
- Minimum values
- Maximum values
- Statistics
Example:Query:
SELECT * FROM orders WHERE amount > 1000
Without Predicate Pushdown:
Read entire Parquet file->Load all data into memory->Apply filter amount > 1000

With Predicate Pushdown:
Check Parquet metadata->Skip irrelevant data blocks->Read only required rows/columns->Load filtered data into memory
Predicate Pushdown helps by:
- Reducing data read from disk
- Loading less data into memory
- Reducing network transfer
- Improving query execution speed
Example:
If a Parquet file contains 100 GB data but the filter needs only 5 GB,
Spark loads only the required portion instead of the complete file.
Therefore, Predicate Pushdown improves Spark performance by minimizing unnecessary data scanning and memory usage.

Q10: Write a code snippet to add a new column final_price which is the base_price multiplied by 1.18 (18% tax). 

In [0]:
from pyspark.sql.functions import col
final_price_df=df.withColumn(
    "final_price",
    col("base_price") * 1.18
)
final_price_df.select(
    "product_id",
    "base_price",
    "final_price"
).show(10)

+----------+----------+------------------+
|product_id|base_price|       final_price|
+----------+----------+------------------+
|     P1001|   1299.29|         1533.1622|
|     P1002|  10970.97|12945.744599999998|
|     P1003|  13929.67|16437.010599999998|
|     P1004|  22998.27|27137.958599999998|
|     P1005|   33080.1|         39034.518|
|     P1006|  13385.54|        15794.9372|
|     P1007|  11490.95|         13559.321|
|     P1008|  44239.92|52203.105599999995|
|     P1009|  11005.07|        12985.9826|
|     P1010|  19111.88|        22552.0184|
+----------+----------+------------------+
only showing top 10 rows


Q11: What is the difference between Transformations and Actions? Provide two examples of each. 

Ans:
Transformations are operations that create a new DataFrame/RDD from an existing DataFrame/RDD.
They do not execute immediately because Spark follows Lazy Evaluation.
Transformations only build the DAG execution plan.
Execution happens when an Action is called.
Transformations:
1.filter()
Used to select rows based on a condition.
Example:
df.filter(df.amount > 1000)
2.select()
Used to choose specific columns.
Example:
df.select("product_id","price")
Actions
Actions trigger the actual execution of transformations.
When an action is called:
DAG is executed->Tasks are created->Executors process the data->Result is returned
Examples of Actions:
1.show()
Displays records from a DataFrame.
Example:
df.show()
2.count()
Returns the total number of records.
Example:
df.count()
Example Flow:
df.filter()->df.select()->DAG creation->df.show()->Spark execution starts.
Therefore, transformations define what operations should be performed, while actions execute those operations and produce results.

Q12: Write the Spark command to load a Parquet file from "path/to/input", filter out any rows where user_id is null, and save the result as a CSV at "path/to/output". 

In [0]:
input_df = spark.read.parquet(
    "path/to/input"
)
clean_df = input_df.filter(
    input_df.user_id.isNotNull()
)

clean_df.write \
.mode("overwrite") \
.option("header", True) \
.csv("path/to/output")

In [0]:
clean_df = df.filter(
    df.user_id.isNotNull()
)
clean_df.show(10)

+--------+-------+----------+------------+--------------+---------+----------+--------+---------+---------+------+--------+-------------+----------+------------+
|order_id|user_id|product_id|    old_name|      category|    price|base_price|quantity|   amount|   status|region|priority|customer_name|order_date|discount_pct|
+--------+-------+----------+------------+--------------+---------+----------+--------+---------+---------+------+--------+-------------+----------+------------+
|  100001|  20001|     P1001|       Novel|         Books|  1299.29|   1299.29|       4|  5197.16|  Pending| North|     Low|  Pooja Verma|2025-03-08|           0|
|  100002|  20002|     P1002|      Laptop|   Electronics| 10970.97|  10970.97|       1| 10970.97|  Pending|  West|    High| Rohan Sharma|2024-06-12|          15|
|  100003|  20003|     P1003|Coffee Beans|       Grocery| 13929.67|  13929.67|       6| 83578.02|Completed| North|  Medium|  Priya Singh|2024-09-27|          20|
|  100004|  20004|     P1004

Q13: In Spark Architecture, what is the difference between Client Mode and Cluster Mode? 

In Apache Spark, Client Mode and Cluster Mode define where the Driver Program runs during Spark application execution.
In Client Mode, the Spark Driver runs on the client machine where the application is submitted.
The client machine communicates with the Cluster Manager and Executors.
Architecture Flow:
Client Machine
↓
Driver Program
↓
Cluster Manager
↓
Worker Nodes
↓
Executors
Working:
1. User submits Spark application.
2. Driver starts on the client machine.
3. Cluster Manager allocates resources.
4. Executors are launched on Worker Nodes.
5. Driver sends tasks to Executors.
Ex:
Running Spark from a laptop:
Laptop → Driver
Cluster → Executors
Advantages:
- Easy debugging
- Useful for development and testing
- Driver logs are available locally
Disadvantage:
If the client machine disconnects, the Spark application may fail.
In Cluster Mode, the Spark Driver runs inside the cluster on one of the Worker Nodes.
The client only submits the application and is no longer required.
Architecture Flow:
Client
↓
Cluster Manager
↓
Worker Node (Driver)
↓
Executors
 Working:
1. User submits Spark application.
2. Cluster Manager starts Driver inside the cluster.
3. Executors are created on Worker Nodes.
4. Driver manages task execution.

Advantages:
- More reliable
- Suitable for production workloads
- Client machine can disconnect after submission
Therefore, Client Mode is mainly used for development, while Cluster Mode is preferred for production Spark applications.

Q14: Write a query to filter a dataset for rows where the region is 'North' OR the priority is 'High'. 

In [0]:
filtered_df = df.filter(
    (df.region == "North") |
    (df.priority == "High")
)

filtered_df.show(10)

+--------+-------+----------+--------------+--------------+---------+----------+--------+---------+---------+------+--------+-------------+----------+------------+
|order_id|user_id|product_id|      old_name|      category|    price|base_price|quantity|   amount|   status|region|priority|customer_name|order_date|discount_pct|
+--------+-------+----------+--------------+--------------+---------+----------+--------+---------+---------+------+--------+-------------+----------+------------+
|  100001|  20001|     P1001|         Novel|         Books|  1299.29|   1299.29|       4|  5197.16|  Pending| North|     Low|  Pooja Verma|2025-03-08|           0|
|  100002|  20002|     P1002|        Laptop|   Electronics| 10970.97|  10970.97|       1| 10970.97|  Pending|  West|    High| Rohan Sharma|2024-06-12|          15|
|  100003|  20003|     P1003|  Coffee Beans|       Grocery| 13929.67|  13929.67|       6| 83578.02|Completed| North|  Medium|  Priya Singh|2024-09-27|          20|
|  100005|  2000

Q15: When exploring a dataset, why is it safer to use .show(5) instead of .collect() on a multi-terabyte dataset? 

In Apache Spark, both show() and collect() are actions used to view data, but they behave differently.
show(5) displays only the first 5 rows of a DataFrame.
It retrieves a limited amount of data from Executors to the Driver.
Ex:
df.show(5)
Execution:
Large Dataset
↓
Executors process data
↓
Only 5 rows sent to Driver
Advantages:
- Loads only a small sample of data
- Uses less Driver memory
- Faster for data exploration
- Safe for very large datasets
collect()
collect() retrieves the entire dataset from all Executors and brings it to the Driver memory.
Ex:
df.collect()
Execution:
Large Dataset
↓
Executors process all partitions
↓
Entire data sent to Driver
Problems with multi-terabyte datasets:
- Huge network transfer
- High Driver memory usage
- Slow execution
- Can cause Driver Out Of Memory (OOM) failure
Ex:
Dataset size:
5 TB
Using:
df.collect()
Spark tries to move all 5 TB data into Driver memory.
Result:
Driver may crash due to insufficient memory.

Using:
df.show(5)
Spark fetches only 5 rows.
The application remains stable.
Therefore, .show(5) is safer because it limits the amount of data transferred to the Driver and prevents memory failures while exploring large Spark datasets.